In [1]:
import pandas as pd
from datasets import load_dataset

## Load Dataset

In [2]:
ds = load_dataset('neurovlm/pubmed_summary_qa', split='train')
ds

Dataset({
    features: ['pmid', 'doi', 'title', 'summary', 'question_1', 'answer_1', 'question_2', 'answer_2', 'question_3', 'answer_3'],
    num_rows: 29146
})

In [3]:
ds[0]

{'pmid': 10022492,
 'doi': '10.1093/cercor/9.1.20',
 'title': 'Working Memory Capacity Constraints',
 'summary': 'Working memory, the ability to hold and manipulate information, has limited capacity. The physiological basis of this limitation is not well understood. Working memory capacity is influenced by the dorsolateral prefrontal cortex (DLPFC) and other brain regions, which exhibit distinct responses to increasing cognitive load. These responses can reflect capacity constraints, providing insight into the physiological characteristics of working memory.',
 'question_1': 'What is working memory and what are its limitations?',
 'answer_1': 'Working memory refers to the ability to hold and manipulate information in mind for a short period. It has limited capacity, meaning it can only handle a certain amount of information at a time.',
 'question_2': 'Which brain region is involved in working memory capacity constraints?',
 'answer_2': "The dorsolateral prefrontal cortex (DLPFC) is a 

## Convert to DataFrame

In [4]:
raw_df = ds.to_pandas()
print(f'Shape: {raw_df.shape}')
print(f'Columns: {raw_df.columns.tolist()}')
raw_df.head()

Shape: (29146, 10)
Columns: ['pmid', 'doi', 'title', 'summary', 'question_1', 'answer_1', 'question_2', 'answer_2', 'question_3', 'answer_3']


,pmid,doi,title,summary,question_1,answer_1,question_2,answer_2,question_3,answer_3
0,10022492,10.1093/cercor/9.1.20,Working Memory Capacity Constraints,"Working memory, the ability to hold and manipu...",What is working memory and what are its limita...,Working memory refers to the ability to hold a...,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
1,10022494,10.1093/cercor/9.1.35,Visuomotor Task Brain Activity,Visuomotor tasks involve the transformation of...,What is a visuomotor task?,A visuomotor task is a type of task that requi...,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
2,10022496,10.1093/cercor/9.1.65,Auditory system responses to tones,The auditory system is a complex network that ...,What is the auditory system?,The auditory system is a complex network of br...,How does the auditory system respond to differ...,The auditory system's response to sound varies...,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
3,10051677,10.1073/pnas.96.5.2532,Thirst and cerebral blood flow,Thirst is a complex physiological state that i...,What brain regions are involved in thirst regu...,"The anterior cingulate region, middle temporal...",How does plasma sodium concentration affect th...,Changes in plasma sodium concentration are cor...,What happens to cerebral blood flow during thi...,Cerebral blood flow increases in certain regio...
4,10191322,10.1523/JNEUROSCI.19-08-03050.1999,Bilingual Language Processing,Bilingual language processing involves the cor...,What brain regions are involved in language pr...,Language processing involves areas in the pref...,How does bilingualism affect language processi...,Bilingualism is associated with overlapping ac...,What is functional magnetic resonance imaging ...,Functional magnetic resonance imaging (fMRI) i...


In [5]:
raw_df.columns.tolist()

['pmid',
 'doi',
 'title',
 'summary',
 'question_1',
 'answer_1',
 'question_2',
 'answer_2',
 'question_3',
 'answer_3']

## Transform to question + answer format

In [6]:
# Dataset has question_1/answer_1, question_2/answer_2, question_3/answer_3 columns
# Unpivot all 3 QA pairs per row into individual rows
records = []

for _, row in raw_df.iterrows():
    for i in range(1, 4):
        q = row.get(f'question_{i}')
        a = row.get(f'answer_{i}')
        if pd.notna(q) and pd.notna(a):
            q = str(q).strip()
            a = str(a).strip()
            if q and a:
                records.append({'question': q, 'answer': a})

df = pd.DataFrame(records)
print(f'Total QA pairs: {len(df)}')
df.head(10)

Total QA pairs: 87438


,question,answer
0,What is working memory and what are its limita...,Working memory refers to the ability to hold a...
1,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
2,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
3,What is a visuomotor task?,A visuomotor task is a type of task that requi...
4,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
5,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...
6,What is the auditory system?,The auditory system is a complex network of br...
7,How does the auditory system respond to differ...,The auditory system's response to sound varies...
8,What is tonotopic organization in the auditory...,Tonotopic organization refers to the mapping o...
9,What brain regions are involved in thirst regu...,"The anterior cingulate region, middle temporal..."


## Clean and filter

In [7]:
# Remove duplicates
df = df.drop_duplicates(subset=['question']).reset_index(drop=True)
print(f'After dedup: {len(df)}')

# Remove very short answers (less than 200 chars) — not useful for text-to-text eval
df = df[df['answer'].str.len() >= 200].reset_index(drop=True)
print(f'After filtering short answers: {len(df)}')

# Remove very short questions
df = df[df['question'].str.len() >= 10].reset_index(drop=True)
print(f'After filtering short questions: {len(df)}')

df.head()

After dedup: 60397
After filtering short answers: 50558
After filtering short questions: 50558


,question,answer
0,Which brain region is involved in working memo...,The dorsolateral prefrontal cortex (DLPFC) is ...
1,Are other brain regions also involved in worki...,"Yes, other brain regions, such as the premotor..."
2,What is a visuomotor task?,A visuomotor task is a type of task that requi...
3,What brain regions are involved in visuomotor ...,The brain regions involved in visuomotor trans...
4,What is the role of the prefrontal cortex in v...,The prefrontal cortex is involved in the prepa...


## Inspect

In [8]:
print(f'Question length — mean: {df["question"].str.len().mean():.0f}, '
      f'median: {df["question"].str.len().median():.0f}')
print(f'Answer length   — mean: {df["answer"].str.len().mean():.0f}, '
      f'median: {df["answer"].str.len().median():.0f}')
print(f'\nSample question:\n{df.iloc[0]["question"]}')
print(f'\nSample answer:\n{df.iloc[0]["answer"]}')

Question length — mean: 58, median: 58
Answer length   — mean: 259, median: 252

Sample question:
Which brain region is involved in working memory capacity constraints?

Sample answer:
The dorsolateral prefrontal cortex (DLPFC) is a key brain region involved in working memory capacity constraints, exhibiting an 'inverted-U' shaped neurophysiological response to increasing cognitive load.


## Save Data

In [9]:
df.to_csv('pubmed_summary_qa.csv', index=False)
print(f'Saved {len(df)} rows to pubmed_summary_qa.csv')

Saved 50558 rows to pubmed_summary_qa.csv
